# Phase 3 — Rhetorical Role Prediction — Model 4 — LoRA

Run this notebook from top to bottom in a fresh kernel with the matching foundation checkpoint and tokenizer. The experiment is already selected in the configuration cell.

**Source:** `phase3/capstone-p3-model-4-and-6.ipynb` at commit `7e7c8a9f1605c5f9836a3cb2cb7c28004343f5f3`. Model/training/evaluation/submission implementations are copied unchanged; only configuration selections and output directories differ. Saved outputs and execution counts were cleared during organization.

**Outputs:** use a separate Kaggle session or working directory for each run. The original submission filenames are preserved, so some ZIP files at the working-directory root can be overwritten by another run.

## Original implementation notes

# Phase 3 Rhetorical Roles - Phase-1 model only (Model 4 or Model 6; full / LoRA / LoRMA)
Sentence-level rhetorical-role classification using **only your Phase-1 `EfficientLegalLM`**. There is no InLegalBERT, no second model and no blending in this notebook.

* Choose the Phase-1 model with `PH1_OPTION` in the config cell: **1** = Model 4 (16 layers, 768 hidden, GQA 12Q/2KV, 256-token sliding window) or **2** = Model 6 (20 layers, 1,024 hidden, GQA 16Q/4KV, 512-token sliding window).
* Choose how it is fine-tuned with `FUSE_METHOD`: `"full"` (all weights trained), `"lora"` (frozen base + additive low-rank `A/B`), `"lorma"` (frozen base + multiplicative low-rank update) or `"kd"` (knowledge distillation: the fully fine-tuned model is the teacher, a half-depth student initialised from its pre-trained layers learns from hard labels + the teacher's soft labels; the student makes the submission).
* The checkpoint `best.pt` and your BPE tokenizer are loaded exactly as in the Phase-2 notebook; attention is bidirectional at fine-tune time and a new classification head is added.

Input per sentence: the target sentence plus a window of neighbouring sentences, a domain tag and a relative-position bucket. The classifier produces a probability over the 13 labels per sentence and the argmax is the predicted label. Early stopping uses the mean of CL and IT dev macro-F1. Dev scores are printed and saved in `phase1_report.json`; `phase1_logits.npz` keeps the raw dev/test logits.

Output: `/kaggle/working/submission_rr_phase1_opt<PH1_OPTION>_<method>.zip` containing `prediction.json` (`{doc_id: [label, ...]}`).

**Attach as Kaggle inputs:** the checkpoint dataset (`best.pt`) of the chosen Phase-1 model and the tokenizer dataset (`tokenizer.json`). Internet is only needed to download the competition data from the Hugging Face dataset repo.

In [ ]:
import os, sys, json, math, random, time, zipfile, glob, re, gc
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import precision_recall_fscore_support, classification_report

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Logged in.")

## Config

In [ ]:
# ===================== what to run =====================
RETRAIN_LORA   = True            # True: fine-tune the Phase-1 model on the task; False: load LORA_CKPT
LORA_CKPT      = None            # a best_<method>_rr.pt saved by an earlier run of this notebook (used when RETRAIN_LORA=False)

# ===================== data / context =====================
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P3 = "Phase_3_RR"
PREV_K, NEXT_K = 3, 1            # neighbouring sentences given as context
POS_BUCKETS    = 20              # relative position of the sentence in its document
USE_DOMAIN_TAG = True
CLASS_WEIGHT_POWER = 0.5         # 0 = plain CE, 0.5 = inverse sqrt frequency, 1 = inverse frequency

# ===================== Phase-1 model fine-tuning =====================
FUSE_METHOD    = 'lora'          # <<< CHOOSE: "full" | "lora" | "lorma" | "kd"  (how the Phase-1 model is fine-tuned)
BIDIRECTIONAL  = True
BIDIR_WINDOW   = 256
RANK, ALPHA    = 8, 16.0
A_CTX_TOKENS   = 48              # each context sentence is clipped to this many BPE tokens
A_TGT_TOKENS   = 200             # target sentence clip
A_MAX_LEN      = 384
A_INFER_BATCH  = 64
A_PATIENCE     = 3
A_HEAD_LR, A_WD, A_WARMUP = 5e-4, 0.01, 0.06
# per-method presets: backbone LR, epochs, batch, gradient checkpointing (override below if you like)
_PRESETS = {
    "lora":  dict(lr=3e-4, epochs=8, batch=16, gc=False),
    "lorma": dict(lr=3e-4, epochs=8, batch=16, gc=False),
    "full":  dict(lr=5e-5, epochs=8, batch=16, gc=True),
    "kd":    dict(lr=5e-5, epochs=8, batch=16, gc=True),    # student: all weights train; the teacher uses the "full" preset    # every weight trains: lower LR, checkpointing to fit the T4
}
assert FUSE_METHOD in _PRESETS, "FUSE_METHOD must be 'full', 'lora', 'lorma' or 'kd'"
METHOD = FUSE_METHOD
A_LR, A_EPOCHS, A_BATCH, USE_GC = (_PRESETS[FUSE_METHOD][k] for k in ("lr", "epochs", "batch", "gc"))

SEED  = 42
SMOKE = False                    # True = tiny run to check the whole pipeline end to end

# ----------------------------- architecture of the Phase-1 model --
PH1_OPTION = 1                   # <<< CHOOSE: 1 = Model 4 (EfficientLegalLM, 768d / 16 layers) | 2 = Model 6 (1024d / 20 layers)
assert PH1_OPTION in (1, 2), "PH1_OPTION must be 1 or 2"
if PH1_OPTION == 1:
    D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS = 768, 16, 12, 2
    HEAD_DIM, D_FF, MAX_SEQ_LEN = D_MODEL // N_HEADS, 2048, 1024
    ROPE_BASE, WINDOW_SIZE = 10000.0, 256
else:   # the second Phase-1 model
    D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS = 1024, 20, 16, 4
    HEAD_DIM, D_FF, MAX_SEQ_LEN = D_MODEL // N_HEADS, 2432, 1024
    ROPE_BASE, WINDOW_SIZE = 10000.0, 512

# ===================== knowledge distillation (FUSE_METHOD = "kd") =====================
# stage 1: the full-depth model is fully fine-tuned as the TEACHER.
# stage 2: a half-depth STUDENT (initialised from evenly spaced pre-trained layers) is trained on
#          (1-KD_ALPHA) * class-weighted CE(hard labels) + KD_ALPHA * T^2 * KL(teacher soft labels || student). The student makes the submission.
KD_TEACHER_CKPT   = None         # best_teacher_rr.pt from an earlier run -> skips teacher training
KD_TEACHER_LR     = 5e-5
KD_TEMP           = 2.0
KD_ALPHA          = 0.5
KD_STUDENT_LAYERS = N_LAYERS // 2
KD_LAYER_MAP      = [int(round(x)) for x in np.linspace(0, N_LAYERS - 1, KD_STUDENT_LAYERS)]
assert len(set(KD_LAYER_MAP)) == KD_STUDENT_LAYERS
print(f"Phase-1 option {PH1_OPTION} | method: {FUSE_METHOD} | backbone LR {A_LR} | epochs {A_EPOCHS} | batch {A_BATCH} | grad-checkpointing {USE_GC}")

CKPT_PATH = None
TOK_DIR   = None
OUT_DIR = '/kaggle/working/phase3_model4_lora'
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = dev.type == "cuda"
print("device:", dev, "|", torch.cuda.get_device_name(0) if use_amp else "")

## Locate Phase-1 checkpoint & tokenizer (as in the Phase-2 notebook)

In [ ]:
from transformers import PreTrainedTokenizerFast


def _find(patterns, prefer=()):
    hits = sorted({h for p in patterns for h in glob.glob(p, recursive=True)})
    for key in prefer:
        pref = [h for h in hits if key in h.lower()]
        if pref:
            return pref[0], hits
    return (hits[0] if hits else None), hits


if CKPT_PATH is None:
    CKPT_PATH, hits = _find(["/kaggle/input/**/best.pt"], prefer=(("m4",) if PH1_OPTION == 1 else ("m6",)) + ("capstone",))
    print("best.pt candidates:", hits)
    print(f"PH1_OPTION={PH1_OPTION}: make sure the checkpoint belongs to Model {4 if PH1_OPTION == 1 else 6}")
if TOK_DIR is None:
    tj, hits = _find(["/kaggle/input/**/tokenizer.json"], prefer=("legal_tokenizer", "tokenizer"))
    TOK_DIR = os.path.dirname(tj) if tj else None
    print("tokenizer.json candidates:", hits)
assert CKPT_PATH and os.path.isfile(CKPT_PATH), "best.pt not found - attach the checkpoint dataset or set CKPT_PATH"
assert TOK_DIR and os.path.isdir(TOK_DIR), "tokenizer not found - attach the tokenizer dataset or set TOK_DIR"
print("using checkpoint:", CKPT_PATH)
print("using tokenizer :", TOK_DIR)

tok_L = PreTrainedTokenizerFast.from_pretrained(TOK_DIR)
tok_L.model_max_length = 10 ** 9          # we window ourselves; silence the length warning
VOCAB_SIZE = len(tok_L)
print(f"VOCAB_SIZE={VOCAB_SIZE} EOS={tok_L.eos_token_id} PAD={tok_L.pad_token_id} fast={tok_L.is_fast}")

PAD_L = tok_L.pad_token_id if tok_L.pad_token_id is not None else 0
SEP_L = tok_L.eos_token_id if tok_L.eos_token_id is not None else PAD_L

## Data

In [ ]:
def fetch(rel):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)

def read_json(rel):
    with open(fetch(rel), "r", encoding="utf-8") as fh:
        return json.load(fh)

def as_doc_list(data):
    """train/dev: list of {id, text, labels}; test: dict {doc_id: [sentences]}. Returns list of {id, text, labels?}."""
    if isinstance(data, dict):
        if "text" in data and isinstance(data["text"], list):
            data = [data]
        else:
            items = []
            for k, v in data.items():
                if isinstance(v, list):
                    items.append({"id": k, "text": v})
                elif isinstance(v, dict):
                    v = dict(v); v.setdefault("id", k)
                    items.append(v)
            data = items
    docs = []
    for d in data:
        did = str(d.get("id", d.get("doc_id", d.get("document_id"))))
        text = d.get("text", d.get("sentences"))
        docs.append({"id": did, "text": list(text), "labels": d.get("labels")})
    return docs

vocab_raw = read_json(f"{P3}/Train/label_vocab.json")
if isinstance(vocab_raw, dict) and all(isinstance(v, int) for v in vocab_raw.values()):
    LABELS = [k for k, _ in sorted(vocab_raw.items(), key=lambda kv: kv[1])]
elif isinstance(vocab_raw, dict) and all(isinstance(k, str) and k.isdigit() for k in vocab_raw):
    LABELS = [vocab_raw[k] for k in sorted(vocab_raw, key=int)]
elif isinstance(vocab_raw, dict) and "labels" in vocab_raw:
    LABELS = list(vocab_raw["labels"])
elif isinstance(vocab_raw, dict):
    LABELS = list(vocab_raw.keys())
else:
    LABELS = list(vocab_raw)
LABELS = [str(l) for l in LABELS]
L2I = {l: i for i, l in enumerate(LABELS)}
K = len(LABELS)
print(K, "labels:", LABELS)

splits = {}
for dom in ("CL", "IT"):
    for sp in ("train", "dev", "test"):
        folder = "Test" if sp == "test" else "Train"
        splits[(dom, sp)] = as_doc_list(read_json(f"{P3}/{folder}/{dom}_{sp}.json"))
        docs = splits[(dom, sp)]
        print(f"{dom} {sp:5s}: {len(docs):4d} docs, {sum(len(d['text']) for d in docs):6d} sentences")
        if sp != "test":
            for d in docs:
                assert d["labels"] is not None and len(d["labels"]) == len(d["text"]), d["id"]
                for l in d["labels"]:
                    assert l in L2I, f"unknown label {l!r}"
if SMOKE:
    for key in splits: splits[key] = splits[key][:3]
    A_EPOCHS = 1
cnt = Counter(l for dom in ("CL", "IT") for d in splits[(dom, "train")] for l in d["labels"])
print("train label counts:", {l: cnt[l] for l in LABELS})

def pos_bucket(i, n):
    return min(POS_BUCKETS - 1, int(POS_BUCKETS * i / max(n, 1)))

def head_text(dom, i, n):
    return (f"{dom} " if USE_DOMAIN_TAG else "") + f"position {pos_bucket(i, n)} of {POS_BUCKETS}."

## Shared training / evaluation helpers

In [ ]:
def macro_scores(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=list(range(K)), average="macro", zero_division=0)
    return float(p), float(r), float(f)

def class_weights(y):
    freq = np.bincount(y, minlength=K).astype(float) + 1.0
    w = 1.0 / freq ** CLASS_WEIGHT_POWER
    w = w / (w * freq).sum() * freq.sum()          # average per-example weight stays 1
    return torch.tensor(w, dtype=torch.float32, device=dev)

def make_sched(opt, total_steps, warm_frac, floor):
    warm = max(1, int(warm_frac * total_steps))
    return torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(floor, (total_steps - s) / max(1, total_steps - warm)))

@torch.no_grad()
def predict_logits(model, collate_fn, ex, bs):
    model.eval()
    order = np.argsort([len(x) for x in ex["ids"]])
    out = np.zeros((len(order), K), dtype=np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        *inp, _y = collate_fn(ex, idx)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            lg = model(*inp)
        out[idx] = lg.float().cpu().numpy()
    return out

def evaluate_dev(model, collate_fn, dev_ex, bs):
    res = {}
    for dom in ("CL", "IT"):
        res[dom] = macro_scores(dev_ex[dom]["y"], predict_logits(model, collate_fn, dev_ex[dom], bs).argmax(1))
    res["mean_f1"] = (res["CL"][2] + res["IT"][2]) / 2
    return res

def run_training(model, opt, sched, scaler, train_ex, dev_ex, collate_fn, ckpt_file, epochs, patience, batch, infer_bs, cls_w, tag, teacher_logits=None, kd_T=2.0, kd_alpha=0.5):
    """Early-stopped training; model selection on the mean of CL and IT dev macro-F1."""
    trainable = [p for p in model.parameters() if p.requires_grad]
    best, bad, t0 = -1.0, 0, time.time()
    N = len(train_ex["y"])
    for ep in range(1, epochs + 1):
        model.train()
        order = np.random.permutation(N); tot, nb = 0.0, 0
        bar = tqdm(range(0, N, batch), desc=f"{tag} ep{ep}/{epochs}", leave=False)
        for bi in bar:
            idx = order[bi:bi + batch]
            *inp, y = collate_fn(train_ex, idx)
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                lg = model(*inp)
            loss = F.cross_entropy(lg.float(), y, weight=cls_w)
            if teacher_logits is not None:                 # knowledge distillation: blend with the teacher's soft labels
                tl = torch.from_numpy(teacher_logits[idx]).to(dev)
                kd = F.kl_div(F.log_softmax(lg.float() / kd_T, -1), F.softmax(tl / kd_T, -1), reduction="batchmean") * kd_T * kd_T
                loss = (1.0 - kd_alpha) * loss + kd_alpha * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss.detach()); nb += 1
            bar.set_postfix(loss=f"{tot / nb:.4f}")
        r = evaluate_dev(model, collate_fn, dev_ex, infer_bs)
        print(f"  [{tag}] ep {ep}: loss {tot / nb:.4f} | CL P/R/F1 {r['CL'][0]:.3f}/{r['CL'][1]:.3f}/{r['CL'][2]:.3f} "
              f"| IT P/R/F1 {r['IT'][0]:.3f}/{r['IT'][1]:.3f}/{r['IT'][2]:.3f} | mean F1 {r['mean_f1']:.4f} | {(time.time() - t0) / 60:.1f} min")
        if r["mean_f1"] > best:
            best, bad = r["mean_f1"], 0
            torch.save({"model": model.state_dict(), "dev_f1": best, "epoch": ep}, ckpt_file)
        else:
            bad += 1
            if bad >= patience:
                print("  early stop"); break
    model.load_state_dict(torch.load(ckpt_file, map_location=dev)["model"])
    return best

def stage_outputs(model, collate_fn, dev_ex, test_ex, bs, tag):
    """Dev/test logits of a finished model + its stand-alone dev scores."""
    out = {"dev": {d: predict_logits(model, collate_fn, dev_ex[d], bs) for d in ("CL", "IT")},
           "test": {d: predict_logits(model, collate_fn, test_ex[d], bs) for d in ("CL", "IT")}}
    sc = {d: macro_scores(dev_ex[d]["y"], out["dev"][d].argmax(1)) for d in ("CL", "IT")}
    sc["mean_f1"] = (sc["CL"][2] + sc["IT"][2]) / 2
    out["score"] = sc
    print(f"[{tag}] stand-alone dev  CL P/R/F1 {sc['CL'][0]:.3f}/{sc['CL'][1]:.3f}/{sc['CL'][2]:.3f} | "
          f"IT {sc['IT'][0]:.3f}/{sc['IT'][1]:.3f}/{sc['IT'][2]:.3f} | mean F1 {sc['mean_f1']:.4f}")
    return out

## Phase-1 model definition + checkpoint (code reused from your first notebook)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).type_as(x) * self.weight


def precompute_rope_freqs(dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, freqs_cos, freqs_sin):
    x_r = x.reshape(*x.shape[:-1], -1, 2)
    x0, x1 = x_r[..., 0], x_r[..., 1]
    seq_len = x.shape[2]
    cos = freqs_cos[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    sin = freqs_sin[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    out0 = x0 * cos - x1 * sin
    out1 = x0 * sin + x1 * cos
    return torch.stack([out0, out1], dim=-1).reshape_as(x)


class SwiGLU_FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    B, H, T, D = x.shape
    return x[:, :, None, :, :].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.n_heads = n_heads; self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim; self.n_rep = n_heads // n_kv_heads
        self.use_window = use_window; self.window_size = window_size
        self.attn_p = dropout
        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, freqs_cos, freqs_sin)
        k = apply_rope(k, freqs_cos, freqs_sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v) if use_cache else None
        dp = self.attn_p if self.training else 0.0
        if past_kv is not None:
            if self.use_window:
                k = k[:, :, -self.window_size:]; v = v[:, :, -self.window_size:]
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp)
        else:
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            if self.use_window and T > 1:
                row = torch.arange(T, device=x.device).unsqueeze(1)
                col = torch.arange(T, device=x.device).unsqueeze(0)
                mask = (col <= row) & (col > row - self.window_size)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp)
            else:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dp)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.out_proj(out)), new_kv


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAttention(d_model, n_heads, n_kv_heads, head_dim, dropout, use_window, window_size)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU_FFN(d_model, d_ff, dropout)

    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        attn_out, new_kv = self.attn(self.attn_norm(x), freqs_cos, freqs_sin, use_cache, past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv


class EfficientLegalLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, head_dim, d_ff,
                 max_seq_len, rope_base=10000.0, dropout=0.0, window_size=256, use_gc=True):
        super().__init__()
        self.d_model = d_model; self.n_layers = n_layers; self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len; self.use_gc = use_gc
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout,
                             use_window=(i % 2 == 0), window_size=window_size)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        fc, fs = precompute_rope_freqs(head_dim, max_seq_len, rope_base)
        self.register_buffer("freqs_cos", fc, persistent=False)
        self.register_buffer("freqs_sin", fs, persistent=False)

    def forward(self, input_ids, targets=None, use_cache=False, past_kvs=None):
        B, T = input_ids.shape
        x = self.emb_dropout(self.tok_emb(input_ids))
        past_len = past_kvs[0][0].shape[2] if past_kvs is not None else 0
        fc = self.freqs_cos[past_len:past_len + T]
        fs = self.freqs_sin[past_len:past_len + T]
        presents = [] if use_cache else None
        for i, layer in enumerate(self.layers):
            pkv = past_kvs[i] if past_kvs is not None else None
            x, kv = layer(x, fc, fs, use_cache, pkv)
            if use_cache:
                presents.append(kv)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits[:, :-1, :].contiguous().view(-1, self.vocab_size),
                                   targets[:, 1:].contiguous().view(-1), ignore_index=-100)
        return logits, loss, presents

In [ ]:
base = EfficientLegalLM(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF,
                        MAX_SEQ_LEN, ROPE_BASE, 0.0, WINDOW_SIZE, use_gc=False)
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state = ckpt["model_state_dict"]
# capstone-m4 saved the torch.compile()d model, so keys carry an "_orig_mod." prefix
state = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in state.items()}
missing, unexpected = base.load_state_dict(state, strict=False)
missing = [k for k in missing if k != "lm_head.weight"]      # tied to tok_emb.weight, so a copy is not required
assert not missing and not unexpected, f"checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}"
print(f"loaded {CKPT_PATH}  (pre-training step {ckpt.get('step')}, loss {ckpt.get('loss')})")
del ckpt, state
base = base.to(dev).eval()

# sanity: a correctly loaded LM has a low next-token loss on legal text (random weights give ~ln(vocab) = 10+)
_txt = " ".join(splits[("CL", "train")][0]["text"])
ids0 = torch.tensor([tok_L(_txt, add_special_tokens=False)["input_ids"][:512]], device=dev)
with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
    lm_loss = base(ids0, targets=ids0)[1].item()
print(f"LM sanity loss on a training judgment: {lm_loss:.3f}  (expect roughly 2-4)")
assert lm_loss < 6.0, "LM loss is too high - weights or tokenizer do not match"

## Sentence classifier on the Phase-1 backbone

In [ ]:
def build_masks(T, pad_mask, bidirectional, window):
    """Boolean attention masks (True = may attend), shape (B,1,T,T): one for window layers, one for full layers."""
    r = torch.arange(T, device=pad_mask.device).unsqueeze(1)
    c = torch.arange(T, device=pad_mask.device).unsqueeze(0)
    if bidirectional:
        full = torch.ones(T, T, dtype=torch.bool, device=pad_mask.device)
        win = (c - r).abs() < window
    else:
        full = c <= r
        win = (c <= r) & (c > r - window)
    key = pad_mask[:, None, None, :]
    return full[None, None] & key, win[None, None] & key


def block_forward(blk, x, cos, sin, mask):
    a = blk.attn
    h = blk.attn_norm(x)
    B, T, _ = h.shape
    q = a.q_proj(h).view(B, T, a.n_heads, a.head_dim).transpose(1, 2)
    k = a.k_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    v = a.v_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)
    k = repeat_kv(k, a.n_rep)
    v = repeat_kv(v, a.n_rep)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, T, -1)
    x = x + a.out_proj(out)
    return x + blk.ffn(blk.ffn_norm(x))


class LoRALinear(nn.Module):
    """y = W x + (alpha/r) B A x   (base W frozen, B starts at zero)."""
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f, in_f = base_linear.weight.shape
        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank

    def forward(self, x):
        return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())


class LoRMALinear(nn.Module):
    """LoRMA (multiplicative): y = (I + (alpha/r) B A) W x   (base W frozen, B starts at zero -> identical to Phase 1 at step 0)."""
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f = base_linear.weight.shape[0]
        self.A = nn.Parameter(torch.randn(rank, out_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * ((h @ self.A.t()) @ self.B.t())


def adapt_linears(backbone, kind, rank, alpha):
    wrapper = LoRMALinear if kind == "lorma" else LoRALinear
    n = 0
    for blk in backbone.layers:
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if isinstance(child, nn.Linear):
                    setattr(parent, name, wrapper(child, rank, alpha))
                    n += 1
    return n

In [ ]:
class LoRARR(nn.Module):
    """Phase-1 backbone (full / LoRA / LoRMA, see FUSE_METHOD) used as a bidirectional encoder; head on [mean of target-sentence tokens | mean of all tokens]."""
    def __init__(self, backbone, n_labels, bidirectional, bidir_window, use_gc):
        super().__init__()
        self.backbone, self.bidirectional, self.bidir_window, self.use_gc = backbone, bidirectional, bidir_window, use_gc
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(2 * D_MODEL, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def forward(self, ids, pad_mask, tmask):
        m = self.backbone
        T = ids.size(1)
        x = m.tok_emb(ids)
        cos, sin = m.freqs_cos[:T], m.freqs_sin[:T]
        window = self.bidir_window if self.bidirectional else WINDOW_SIZE
        full, win = build_masks(T, pad_mask, self.bidirectional, window)
        for blk in m.layers:
            mask = win if blk.attn.use_window else full
            if self.use_gc and self.training:
                x = grad_checkpoint(block_forward, blk, x, cos, sin, mask, use_reentrant=False)
            else:
                x = block_forward(blk, x, cos, sin, mask)
        h = m.final_norm(x).float()
        tm = tmask.float().unsqueeze(-1); am = pad_mask.float().unsqueeze(-1)
        mean_t = (h * tm).sum(1) / tm.sum(1).clamp(min=1)
        mean_a = (h * am).sum(1) / am.sum(1).clamp(min=1)
        return self.head(self.drop(torch.cat([mean_t, mean_a], dim=-1)))

# ---- inputs for Model A:  head + [SEP] + previous: ctx [SEP] ctx ... [SEP] target sentence: TARGET [SEP] next: ctx
def _enc(texts):
    return tok_L(texts, add_special_tokens=False)["input_ids"]

M_PREV, M_TGT, M_NEXT = _enc([" previous:", " target sentence:", " next:"])
_head_cache = {}

def build_examples_A(docs, dom, with_labels=True):
    ids_l, tm_l, y_l, meta = [], [], [], []
    for d in docs:
        sents, n = d["text"], len(d["text"])
        sid = _enc(sents)
        for i in range(n):
            hk = (dom, pos_bucket(i, n))
            if hk not in _head_cache:
                _head_cache[hk] = _enc([head_text(dom, i, n)])[0]
            prev = []
            for x in sid[max(0, i - PREV_K):i]:
                prev += x[:A_CTX_TOKENS] + [SEP_L]
            nxt = []
            for x in sid[i + 1:i + 1 + NEXT_K]:
                nxt += x[:A_CTX_TOKENS] + [SEP_L]
            tgt = sid[i][:A_TGT_TOKENS]
            pre = _head_cache[hk] + [SEP_L] + M_PREV + prev + M_TGT
            post = [SEP_L] + M_NEXT + nxt
            budget = A_MAX_LEN - len(tgt)
            if len(pre) + len(post) > budget:                    # drop the oldest context / the tail of "next"
                pb = min(len(post), budget // 3)
                post = post[:pb]
                pre = pre[-(budget - pb):] if budget - pb > 0 else []
            seq = pre + tgt + post
            tm = [0] * len(pre) + [1] * len(tgt) + [0] * len(post)
            ids_l.append(seq); tm_l.append(tm)
            y_l.append(L2I[d["labels"][i]] if with_labels else -1)
            meta.append((d["id"], i))
    return {"ids": ids_l, "tm": tm_l, "y": y_l, "meta": meta}

def collate_A(ex, idx):
    L = max(len(ex["ids"][i]) for i in idx)
    ids = torch.full((len(idx), L), PAD_L, dtype=torch.long)
    pm = torch.zeros((len(idx), L), dtype=torch.bool)
    tm = torch.zeros((len(idx), L), dtype=torch.bool)
    for r, i in enumerate(idx):
        n = len(ex["ids"][i])
        ids[r, :n] = torch.tensor(ex["ids"][i]); pm[r, :n] = True; tm[r, :n] = torch.tensor(ex["tm"][i], dtype=torch.bool)
    y = torch.tensor([ex["y"][i] for i in idx], dtype=torch.long)
    return ids.to(dev), pm.to(dev), tm.to(dev), y.to(dev)

def cat_ex(parts):
    out = {k: [] for k in parts[0]}
    for p in parts:
        for k in out: out[k] += p[k]
    return out

trainA = cat_ex([build_examples_A(splits[(dom, "train")], dom) for dom in ("CL", "IT")])
devA = {dom: build_examples_A(splits[(dom, "dev")], dom) for dom in ("CL", "IT")}
testA = {dom: build_examples_A(splits[(dom, "test")], dom, with_labels=False) for dom in ("CL", "IT")}
lens = np.array([len(x) for x in trainA["ids"]])
print(f"[A] train {len(trainA['y'])} | dev { {d: len(v['y']) for d, v in devA.items()} } | test { {d: len(v['ids']) for d, v in testA.items()} }")
print(f"[A] token length: mean {lens.mean():.0f}, p95 {np.percentile(lens, 95):.0f}, max {lens.max()} (cap {A_MAX_LEN})")
assert lens.max() <= A_MAX_LEN <= MAX_SEQ_LEN
print("[A] example input:", tok_L.decode(trainA["ids"][5])[:600].replace("\n", " "))

## Adapt (full / LoRA / LoRMA) and train

In [ ]:
def build_optim(model, n_train, lr):
    bb = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone.")]
    hd = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": hd, "lr": A_HEAD_LR}, {"params": bb, "lr": lr}], weight_decay=A_WD)
    steps = math.ceil(n_train / A_BATCH) * A_EPOCHS
    return opt, make_sched(opt, steps, A_WARMUP, 0.05), torch.amp.GradScaler("cuda", enabled=use_amp)

def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad), sum(p.numel() for p in m.parameters())

def unfreeze_all(m):
    for p in m.parameters():
        p.requires_grad_(True)
    m.lm_head.weight.requires_grad_(True)       # tied to tok_emb; the LM head itself is unused

def make_wrapper(backbone, gc_flag):
    return LoRARR(backbone, K, BIDIRECTIONAL, BIDIR_WINDOW, gc_flag)

def make_student(pre_sd, layer_map):
    """Half-depth EfficientLegalLM: embeddings / final norm copied, layer i <- pre-trained layer layer_map[i] (window/full pattern kept)."""
    st = EfficientLegalLM(VOCAB_SIZE, D_MODEL, len(layer_map), N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF,
                          MAX_SEQ_LEN, ROPE_BASE, 0.0, WINDOW_SIZE, use_gc=False)
    new = {k: v for k, v in pre_sd.items() if not k.startswith("layers.")}
    for i, src in enumerate(layer_map):
        pref = f"layers.{src}."
        for k, v in pre_sd.items():
            if k.startswith(pref):
                new[f"layers.{i}." + k[len(pref):]] = v
    missing, unexpected = st.load_state_dict(new, strict=False)
    assert not [k for k in missing if k != "lm_head.weight"] and not unexpected, (missing[:4], unexpected[:4])
    for i, src in enumerate(layer_map):
        st.layers[i].attn.use_window = (src % 2 == 0)      # same attention type as the source layer
    return st

cwA = class_weights(trainA["y"])
print("P1 class weights:", {l: round(float(x), 2) for l, x in zip(LABELS, cwA.cpu().numpy())})
lora_file = os.path.join(OUT_DIR, f"best_{METHOD}_rr.pt")
SKIP_TRAIN = bool((not RETRAIN_LORA) and LORA_CKPT and os.path.exists(LORA_CKPT))
TEACHER_SCORE, T_TRAIN = None, None

if METHOD == "kd":
    # ---------- stage 1: TEACHER = the full-depth model, every weight fine-tuned ----------
    PRE_SD = {k: v.detach().cpu().clone() for k, v in base.state_dict().items()}   # pristine pre-trained weights for the student
    if not SKIP_TRAIN:
        unfreeze_all(base)
        teacher = make_wrapper(base, _PRESETS["full"]["gc"]).to(dev)
        t_tr, t_tot = n_params(teacher)
        print(f"[teacher] full fine-tuning, {t_tr / 1e6:.2f}M trainable parameters")
        teacher_file = os.path.join(OUT_DIR, "best_teacher_rr.pt")
        if KD_TEACHER_CKPT and os.path.exists(KD_TEACHER_CKPT):
            teacher.load_state_dict(torch.load(KD_TEACHER_CKPT, map_location=dev)["model"])
            print("teacher loaded from", KD_TEACHER_CKPT)
        else:
            t_opt, t_sched, t_scaler = build_optim(teacher, len(trainA["y"]), KD_TEACHER_LR)
            run_training(teacher, t_opt, t_sched, t_scaler, trainA, devA, collate_A, teacher_file, A_EPOCHS, A_PATIENCE,
                         A_BATCH, A_INFER_BATCH, cwA, "teacher")
            del t_opt, t_sched, t_scaler
        TEACHER_SCORE = evaluate_dev(teacher, collate_A, devA, A_INFER_BATCH)
        print(f"[teacher] dev  CL F1 {TEACHER_SCORE['CL'][2]:.3f} | IT F1 {TEACHER_SCORE['IT'][2]:.3f} | mean F1 {TEACHER_SCORE['mean_f1']:.4f}")
        T_TRAIN = predict_logits(teacher, collate_A, trainA, A_INFER_BATCH)          # soft labels for every training sentence
        del teacher; base = None
        gc.collect(); torch.cuda.empty_cache()
    # ---------- stage 2: STUDENT = half the layers, initialised from evenly spaced pre-trained layers ----------
    base = make_student(PRE_SD, KD_LAYER_MAP).to(dev)
    del PRE_SD
    print(f"[student] {len(KD_LAYER_MAP)} layers copied from pre-trained layers {KD_LAYER_MAP}; all weights trainable")
elif METHOD == "full":
    unfreeze_all(base)
    print("full fine-tuning: all Phase-1 weights are trainable")
else:
    for p in base.parameters():
        p.requires_grad_(False)
    print(f"wrapped {adapt_linears(base, METHOD, RANK, ALPHA)} linear layers with {METHOD}")

modelA = make_wrapper(base, USE_GC).to(dev)
tr, tot = n_params(modelA)
print(f"[P1] parameters: trainable {tr / 1e6:.2f}M / total {tot / 1e6:.2f}M ({tr / tot:.1%})")
optA, schedA, scalerA = build_optim(modelA, len(trainA["y"]), A_LR)

if SKIP_TRAIN:
    modelA.load_state_dict(torch.load(LORA_CKPT, map_location=dev)["model"])
    print(f"{METHOD} model loaded from", LORA_CKPT)
else:
    kd_kw = dict(teacher_logits=T_TRAIN, kd_T=KD_TEMP, kd_alpha=KD_ALPHA) if METHOD == "kd" else {}
    run_training(modelA, optA, schedA, scalerA, trainA, devA, collate_A, lora_file, A_EPOCHS, A_PATIENCE, A_BATCH, A_INFER_BATCH, cwA, METHOD, **kd_kw)
OUT_A = stage_outputs(modelA, collate_A, devA, testA, A_INFER_BATCH, METHOD)
META_TEST = {d: testA[d]["meta"] for d in ("CL", "IT")}
META_DEV = {d: devA[d]["meta"] for d in ("CL", "IT")}
Y_DEV = {d: np.array(devA[d]["y"]) for d in ("CL", "IT")}
del modelA, optA, schedA, scalerA, base, trainA, devA, testA
gc.collect(); torch.cuda.empty_cache()


## Dev report

In [ ]:
Y_DEV_PRED = {d: OUT_A["dev"][d].argmax(1) for d in ("CL", "IT")}
final = {d: macro_scores(Y_DEV[d], Y_DEV_PRED[d]) for d in ("CL", "IT")}
final["mean_f1"] = (final["CL"][2] + final["IT"][2]) / 2
print(f"Phase-1 option {PH1_OPTION} ({METHOD}) dev macro-F1:  CL P/R/F1 {final['CL'][0]:.3f}/{final['CL'][1]:.3f}/{final['CL'][2]:.3f} | "
      f"IT {final['IT'][0]:.3f}/{final['IT'][1]:.3f}/{final['IT'][2]:.3f} | mean F1 {final['mean_f1']:.4f}")
print("note: early stopping used this dev set, so the dev score is slightly optimistic.")
if TEACHER_SCORE is not None:
    print(f"KD: teacher (full fine-tune) dev mean F1 {TEACHER_SCORE['mean_f1']:.4f} -> distilled student {final['mean_f1']:.4f}")
for d in ("CL", "IT"):
    print(f"\n===== {d} dev =====")
    print(classification_report(Y_DEV[d], Y_DEV_PRED[d], labels=list(range(K)), target_names=LABELS, zero_division=0, digits=3))

np.savez_compressed(os.path.join(OUT_DIR, "phase1_logits.npz"),
                    **{f"{s}_{d}": OUT_A[s][d] for s in ("dev", "test") for d in ("CL", "IT")},
                    **{f"ydev_{d}": Y_DEV[d] for d in ("CL", "IT")})

## Predict test and write submission

In [ ]:
predictions = {}
for d in ("CL", "IT"):
    pred = OUT_A["test"][d].argmax(1)
    per_doc = {}
    for (did, i), p in zip(META_TEST[d], pred):
        per_doc.setdefault(did, {})[i] = LABELS[int(p)]
    for doc in splits[(d, "test")]:
        labs = [per_doc[doc["id"]][i] for i in range(len(doc["text"]))]
        assert len(labs) == len(doc["text"])
        assert doc["id"] not in predictions, f"doc id {doc['id']} appears in both CL and IT test"
        predictions[doc["id"]] = labs
assert all(l in L2I for v in predictions.values() for l in v)

pred_path = os.path.join(OUT_DIR, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as fh:
    json.dump(predictions, fh, ensure_ascii=False)
zip_path = f"/kaggle/working/submission_rr_phase1_opt{PH1_OPTION}_{METHOD}.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")

with open(os.path.join(OUT_DIR, "phase1_report.json"), "w") as fh:
    json.dump({"ph1_option": PH1_OPTION, "method": METHOD, "dev_mean_f1": final["mean_f1"], "teacher_dev": TEACHER_SCORE,
               "dev_CL": {"precision": final["CL"][0], "recall": final["CL"][1], "f1": final["CL"][2]},
               "dev_IT": {"precision": final["IT"][0], "recall": final["IT"][1], "f1": final["IT"][2]}}, fh, indent=2)
print(f"wrote {zip_path}: {len(predictions)} docs, {sum(len(v) for v in predictions.values())} sentences")
print("test label distribution:", Counter(l for v in predictions.values() for l in v).most_common())